# Chord Recognition: From Training to Continuous-Audio Inference
### Architecture reference and implementation guide

**Project:** `RWC_multitask_CNN_v1` · **Version:** 2026-10-21 · **Scope:** hierarchical CQT72 CNN + transposition-invariant chord language model + second-order hidden semi-Markov model (HSMM).

This notebook documents the architecture, data format, inference algorithms, and training setup used in this repository. It combines explanatory sections with runnable artifact checks and implementation notes. The saved weights and cached feature files are used to validate the implementation.

**End-to-end path**

```text
Audio (MP3/WAV)
    ↓  mono 22,050 Hz → harmonic separation (HPSS)
Continuous log-frequency CQT (216 bins → 72 features)
    ↓  overlapping 72×32 windows (0.2 s between centers)
Hierarchical CNN
    ├─ P(root | audio)  [12]
    ├─ P(quality | root, audio)  [6]
    └─ P(seventh | root, quality, audio)  [4]
    ↓  joint posterior over 12×6×4 = 288 chord states
Segmental Viterbi / second-order HSMM
    ├─ CNN acoustic evidence summed across each chord segment
    ├─ chord-event transitions conditioned on two previous events
    └─ explicit chord-duration distribution
    ↓
Timestamped chord events → CSV / JSON / player
```

**Validation snapshot (15 RWC Popular validation songs; 200 ms grid):** CNN-only full root–quality–seventh (RQS) 58.67%; first-order HSMM 64.87%; second-order HSMM **65.45%**. Chord-boundary F1 within ±0.5 s **0.852** for second-order HSMM. These are development/validation results, **not** untouched-test or standardized MIREX benchmarks.

## 0. Rules for a faithful migration

1. **Do not retrain** to start. Restore `best_model.pt` and `harmonic_model.pkl`, then reproduce cached predictions on one song.
2. **Do not use isolated-chord datasets to train harmonic transitions.** Use actual sequential annotations from RWC songs. GuitarSet/IDMT/shared isolated-chord material can help CNN training where labels are appropriate; they are not language-model progression examples.
3. **Keep train, validation and test splits isolated.** Train on 70 songs, tune on 15 validation songs, evaluate the 15 held-out test songs only after freezing decisions.
4. **No annotation data at inference.** All boundaries and chord labels are inferred from audio. Oracle-event evaluation used earlier is a diagnostic only.
5. **The real HSMM is approximate.** It uses a beam and acoustic candidate pruning; it is *segmental Viterbi*, but not an exact all-state exhaustive dynamic program.
6. **Current audio frontend needs parity validation.** Preserve and regression-test the rolling frontend; do not silently change the feature engineering.
7. **Chords in model state omit extensions, inversions and `N` (no-chord).** The 7-dimensional extension head exists but its training labels were severely imbalanced, so extensions are not in the default decoder.
8. **Security:** Pickle/PyTorch checkpoints should only be loaded from trusted project files.

## 1. Project artifacts and local directory layout

Save the three essential files below into your repository (or into ignored local `artifacts/`):

| Source in Google Drive project | Suggested local path | Purpose |
|---|---|---|
| `seed42_cqt72_hierarchical_aug_v1/best_model.pt` | `artifacts/best_model.pt` | CNN parameters, class labels, normalization |
| `seed42_second_order_hsmm_v1/harmonic_model.pkl` | `artifacts/harmonic_model.pkl` | 0/1/2-order counts, priors, duration samples |
| `seed42_songs100/split_manifest.json` | `artifacts/split_manifest.json` | Fixed seed-42 70/15/15 RWC split |

Optional: `features_cqt72_v1/RWC_PNNN.npz` for segment-level training/evaluation; `seed42_second_order_hsmm_v1/continuous_hsmm_v1/rolling_predictions/` for full-song regression fixtures; `.../evaluation/` for old validation comparison. The user's Beatles and other MP3s are **not part of model weights** and should not be committed without appropriate rights.

Suggested modules for Codex: `chordrec/audio.py`, `chordrec/model.py`, `chordrec/language.py`, `chordrec/hsmm.py`, `chordrec/evaluate.py`, `chordrec/cli.py`; `tests/`, `viewer/`, `artifacts/`, `notebooks/`.

In [ ]:
# Setup — run in Colab or local Jupyter
# In terminal: python -m pip install torch numpy scipy librosa soundfile pandas matplotlib mir_eval remotezip
from pathlib import Path
import json, pickle, time, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import librosa
from scipy.stats import lognorm

DRIVE_PROJECT = Path('/content/drive/MyDrive/RWC_multitask_CNN_v1')
LOCAL_PROJECT = Path.cwd()
ART = LOCAL_PROJECT / 'artifacts'

if (ART / 'best_model.pt').exists():
    CNN_PATH = ART / 'best_model.pt'
    LM_PATH = ART / 'harmonic_model.pkl'
    MANIFEST_PATH = ART / 'split_manifest.json'
    FEATURES_DIR = LOCAL_PROJECT / 'data' / 'features_cqt72_v1'
elif DRIVE_PROJECT.exists():
    CNN_PATH = DRIVE_PROJECT / 'seed42_cqt72_hierarchical_aug_v1' / 'best_model.pt'
    LM_PATH = DRIVE_PROJECT / 'seed42_second_order_hsmm_v1' / 'harmonic_model.pkl'
    MANIFEST_PATH = DRIVE_PROJECT / 'seed42_songs100' / 'split_manifest.json'
    FEATURES_DIR = DRIVE_PROJECT / 'features_cqt72_v1'
else:
    raise FileNotFoundError('Copy checkpoint, language model and manifest into ./artifacts/')

for p in (CNN_PATH, LM_PATH, MANIFEST_PATH):
    assert p.is_file(), f'Missing required artifact: {p}'
    print(f'{p.name:24s} {p.stat().st_size / 1024:.1f} KiB')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| device:', DEVICE)

## 2. Data and target representation

**RWC Popular**: fixed seed-42 split of 100 songs: 70 training, 15 validation, 15 held-out test. Our CQT72 segment cache contains one `(72,32)` acoustic example per annotated chord segment with `X`, `root`, `quality`, `seventh`, `t_start`, `t_end`, and possibly other targets. Verify the actual keys because older feature caches have different schemas.

Base state: `z=(root, quality, seventh)`:

- Root: C, Db, D, Eb, E, F, Gb, G, Ab, A, Bb, B (indices 0…11).
- Quality: `maj`, `min`, `dim`, `aug`, `sus2`, `sus4` (indices 0…5).
- Seventh: `none`, `b7`, `maj7`, `dim7` (indices 0…3).
- Index: `state = root * (6*4) + quality * 4 + seventh`.

This Cartesian product includes unusual chord combinations; a future implementation may mask invalid combinations but cannot change the checkpoint output ordering. External datasets and annotation naming conventions must be converted into this exact vocabulary.

In [ ]:
# Inspect the saved class vocabulary, split and feature schema.
# Trusted local checkpoints only: torch.load(...weights_only=False) deserializes Python objects.
ckpt = torch.load(CNN_PATH, map_location='cpu', weights_only=False)
with LM_PATH.open('rb') as f:
    lm = pickle.load(f)
manifest = json.loads(MANIFEST_PATH.read_text())

QUALITY = list(ckpt['quality'])
SEVENTH = list(ckpt['seventh'])
EXTENSIONS = list(ckpt['extensions'])
MU = float(ckpt['mean'])
STD = float(ckpt['std'])
Q, S = len(QUALITY), len(SEVENTH)
N_STATES = 12 * Q * S
assert (Q, S, N_STATES) == (6, 4, 288)
assert QUALITY == list(lm['quality']) and SEVENTH == list(lm['seventh'])
assert [len(manifest['splits'][p]) for p in ('train','val','test')] == [70,15,15]
print('checkpoint epoch:', ckpt['epoch'], 'validation score:', ckpt['score'])
print('quality:', QUALITY, '| seventh:', SEVENTH)
print('extensions:', EXTENSIONS)
print('mean/std:', MU, STD, '| split:', {k: len(v) for k,v in manifest['splits'].items()})
print('duration samples:', len(lm['durations_seconds']))
if FEATURES_DIR.is_dir():
    sample = next(FEATURES_DIR.glob('RWC_P*.npz'), None)
    if sample:
        with np.load(sample, allow_pickle=False) as z:
            print('sample cache:', sample.name, {k: z[k].shape for k in z.files})

## 3. Hierarchical CNN architecture

The CNN is **not** a single flat 288-class classifier. It has a 12-class root prediction, then rotates the input to a root-relative pitch representation, predicts six chord qualities, and predicts one of four seventh categories conditional on the chosen quality. The separate extension head is conditioned on quality and seventh and is not used in our current HSMM.

Two independent feature encoders use Conv1d over the 32 temporal samples, with 72 CQT-frequency channels. BatchNorm+GELU are followed by adaptive time pooling. The second encoder analyzes a candidate-root-rotated feature tensor.

At *training time*, the project trained with **teacher-forced true root/quality/seventh** for downstream heads. At **inference**, teacher forcing is forbidden: enumerate candidate roots (and qualities) to assemble joint probabilities:

\[
P(r,q,s\mid x)=P(r\mid x)\,P(q\mid x,r)\,P(s\mid x,r,q).
\]

Do not replace this calculation by greedy `argmax(root)→argmax(quality)→argmax(seventh)` if the decoder needs alternatives.

In [ ]:
class Encoder72(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(72, 96, 5, padding=2),
            nn.BatchNorm1d(96), nn.GELU(),
            nn.Conv1d(96, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.Conv1d(128, 128, 3, padding=1),
            nn.BatchNorm1d(128), nn.GELU(),
            nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Dropout(0.30)
        )
    def forward(self, x): return self.net(x)

class HierarchicalCNN72(nn.Module):
    def __init__(self):
        super().__init__()
        self.root_encoder = Encoder72()
        self.rel_encoder = Encoder72()
        self.root_head = nn.Linear(128, 12)
        self.quality_head = nn.Sequential(nn.Linear(128,128), nn.GELU(), nn.Linear(128,Q))
        self.q_embed = nn.Embedding(Q,16)
        self.s_embed = nn.Embedding(S,12)
        self.seventh_head = nn.Sequential(nn.Linear(144,128), nn.GELU(), nn.Linear(128,S))
        self.extension_head = nn.Sequential(nn.Linear(156,128), nn.GELU(), nn.Linear(128,len(EXTENSIONS)))
    def root_logits(self,x): return self.root_head(self.root_encoder(x))
    def rel_features(self,x): return self.rel_encoder(x)
    def quality_logits(self,h): return self.quality_head(h)
    def seventh_logits(self,h,q):
        return self.seventh_head(torch.cat([h,self.q_embed(q)],dim=1))
    def extension_logits(self,h,q,s):
        return self.extension_head(torch.cat([h,self.q_embed(q),self.s_embed(s)],dim=1))
    def forward(self, a, rel, q_true, s_true):
        h = self.rel_features(rel)
        return (self.root_logits(a), self.quality_logits(h),
                self.seventh_logits(h,q_true), self.extension_logits(h,q_true,s_true))

model = HierarchicalCNN72().to(DEVICE)
# strict=True is essential: mismatches mean the implementation is not equivalent to saved training.
model.load_state_dict(ckpt['model'], strict=True)
model.eval()
print('Loaded exact parameter names:', len(ckpt['model']))
print('Model parameters:', sum(p.numel() for p in model.parameters()))

### 3.1 Root-relative rotation

`72 = 6 octaves × 12 pitch classes`. For a candidate root `r`, reorder each octave so pitch class `r` becomes relative class 0, `r+1` relative class 1, etc. This creates a transposition-relative pattern that the quality encoder can classify. Crucially, this is **reindexing pitch classes, not changing the original time grid or frequency resolution**. The code's CQT bin ordering must match the checkpoint's expected pitch-class ordering.

Training uses `(X-MU)/STD` after transformation; compute rotation on **unnormalized** values and normalize the resulting root-relative matrix.

In [ ]:
def rotate_to_root(raw, roots):
    # raw: (B,72,32) on DEVICE, roots: (B,)
    b, _, t = raw.shape
    x = raw.reshape(b, 6, 12, t)
    ix = (roots[:,None] + torch.arange(12,device=raw.device)[None,:]) % 12
    return x.gather(2,ix[:,None,:,None].expand(-1,6,-1,t)).reshape(b,72,t)

@torch.no_grad()
def predict_chord_posteriors(X, batch_size=64, top_roots=4):
    # Return normalized posteriors, with top-root approximation; use 12 for full.
    X = np.asarray(X, dtype=np.float32)
    assert X.ndim==3 and X.shape[1:]==(72,32), X.shape
    outputs=[]
    for a in range(0,len(X),batch_size):
        raw=torch.as_tensor(X[a:a+batch_size],device=DEVICE)
        b=raw.shape[0]
        rp=torch.softmax(model.root_logits((raw-MU)/STD),dim=1)
        top=rp.topk(min(top_roots,12),dim=1).indices
        out=torch.zeros((b,N_STATES),device=DEVICE)
        for i in range(top.shape[1]):
            roots=top[:,i]
            h=model.rel_features((rotate_to_root(raw,roots)-MU)/STD)
            qp=torch.softmax(model.quality_logits(h),dim=1)
            for q in range(Q):
                qs=torch.full((b,),q,dtype=torch.long,device=DEVICE)
                sp=torch.softmax(model.seventh_logits(h,qs),dim=1)
                joint=rp.gather(1,roots[:,None])*qp[:,q:q+1]*sp
                for s in range(S):
                    state=roots*(Q*S)+q*S+s
                    out.scatter_(1,state[:,None],joint[:,s:s+1])
        out=out/out.sum(dim=1,keepdim=True).clamp_min(1e-12)
        outputs.append(out.cpu().numpy())
    return np.concatenate(outputs,axis=0) if outputs else np.empty((0,N_STATES),np.float32)

# Tiny structural smoke test (does not measure accuracy)
with torch.no_grad():
    dummy=np.zeros((2,72,32),np.float32)
    posterior=predict_chord_posteriors(dummy,top_roots=2)
assert posterior.shape == (2,288) and np.allclose(posterior.sum(axis=1),1,atol=1e-5)
print('joint posterior shape:',posterior.shape)

## 4. Training the CNN

The training setup uses the project’s saved seed-42 partitions, hierarchical architecture, data augmentation, and mixed datasets. Optimizer, scheduler, batch sampling, and transposition settings are defined in the training script.

Historical details known from checkpoints/logs:

- RWC 100 songs split 70/15/15; approximately 8,729 RWC training chord segments, 1,908 validation, 1,793 test.
- Additional isolated-chord examples were used to improve the acoustic model, with GuitarSet/IDMT/shared chord caches. **They are excluded from HSMM transition training.**
- Best CQT72 hierarchical checkpoint at **epoch 65**, RWC validation RQS `~0.70964` (segment-aligned inference).
- Quality labels `[maj,min,dim,aug,sus2,sus4]`; seventh `[none,b7,maj7,dim7]`; extensions `[b9,9,#9,11,#11,b13,13]`.
- Extension positives were extremely sparse (e.g. 9, 11 and altered extensions largely absent); do **not** use extension predictions as reliable output.
- Source-domain augmentation and sampling are part of the training configuration.

**Suggested training objective:** cross-entropy of root, quality and seventh heads, with optional weighted BCE for the multi-label extension head; use teacher-forced `q_true` and `s_true` in `forward`, and rotate with the true root. Validate via **non-teacher-forced** joint inference. The sample below illustrates **one optimization step** but intentionally does not prescribe the historical loss weights, optimizer or augmentation scheme.

In [ ]:
# CNN training step BLUEPRINT, not the original training loop.
# Requires a supervised minibatch of cached X, root, quality, seventh, extensions.
import torch.nn.functional as F

def illustrative_cnn_train_step(model, optimizer, X, root, quality, seventh, extensions=None):
    model.train()
    raw=torch.as_tensor(X,dtype=torch.float32,device=DEVICE)
    r=torch.as_tensor(root,dtype=torch.long,device=DEVICE)
    q=torch.as_tensor(quality,dtype=torch.long,device=DEVICE)
    s=torch.as_tensor(seventh,dtype=torch.long,device=DEVICE)
    rel=rotate_to_root(raw,r)
    logits_r,logits_q,logits_s,logits_ext = model((raw-MU)/STD,(rel-MU)/STD,q,s)
    losses={'root':F.cross_entropy(logits_r,r),
            'quality':F.cross_entropy(logits_q,q),
            'seventh':F.cross_entropy(logits_s,s)}
    # Extensions optional; no universal loss weighting assumed.
    if extensions is not None:
        ext=torch.as_tensor(extensions,dtype=torch.float32,device=DEVICE)
        losses['extensions']=F.binary_cross_entropy_with_logits(logits_ext,ext)
    loss=sum(losses.values())
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return {k:float(v.detach()) for k,v in losses.items()}

print('Training-step template defined; existing checkpoint remains unchanged.')

## 5. Continuous audio → CQT72 → rolling windows

The CNN was trained on isolated annotated chords, but deployment needs **audio with unknown boundaries**. The prototype predicts once every `STEP=0.2 s` using a **32-frame (~0.72–0.74 s) window** sampled from the continuous representation. Thus, the CNN sees ~0.74 s of context at each 200 ms observation. The HSMM works over the 200 ms **observation grid**, not 200 ms audio snippets.

Prototype settings: mono resample 22,050 Hz, hop 512, HPSS, CQT with 36 bins per octave / 216 total bins starting at C1, aggregate groups of 3 to 72 bins; take square root, normalize each time slice by its sum. **The exact CQT frequency aggregation and HPSS conventions should be checked against the original feature-extraction notebook.** The prototype computes CQT on overlapping 30-second chunks to cap memory; chunk boundaries may introduce artifacts.

The following implementation is meant for pipeline migration and reproducibility against the previously evaluated prototype. Its 30 s chunked audio preprocessing can be slow on CPU.

In [ ]:
SR=22050
HOP=512
WINDOW_FRAMES=32
STEP=0.2

def read_audio(path):
    y,sr=librosa.load(str(path),sr=SR,mono=True)
    return np.asarray(y,np.float32)

def extract_continuous_cqt72(y, chunk_sec=30.0, overlap_sec=1.0):
    duration=len(y)/SR
    chunks=[]; times=[]
    for begin in np.arange(0,duration,chunk_sec):
        finish=min(duration,begin+chunk_sec)
        lo=max(0.0,begin-overlap_sec)
        hi=min(duration,finish+overlap_sec)
        clip=y[int(lo*SR):int(hi*SR)]
        harmonic,_=librosa.effects.hpss(clip,margin=(2.0,4.0))
        c=np.abs(librosa.cqt(harmonic,sr=SR,hop_length=HOP,
                               fmin=librosa.note_to_hz('C1'),
                               bins_per_octave=36,n_bins=216)).astype(np.float32)
        c=c.reshape(72,3,-1).sum(axis=1)
        ts=librosa.frames_to_time(np.arange(c.shape[1]),sr=SR,hop_length=HOP)+lo
        keep=(ts>=begin)&(ts<finish)
        chunks.append(c[:,keep]); times.append(ts[keep])
    return np.concatenate(chunks,axis=1),np.concatenate(times)

def make_rolling_windows(cqt, frame_times, duration, step=STEP):
    centers=np.arange(step/2,duration,step,dtype=np.float64)
    out=np.empty((len(centers),72,WINDOW_FRAMES),np.float32)
    offsets=np.linspace(-(WINDOW_FRAMES-1)*HOP/(2*SR),
                         (WINDOW_FRAMES-1)*HOP/(2*SR),WINDOW_FRAMES)
    for i,center in enumerate(centers):
        targets=np.clip(center+offsets,frame_times[0],frame_times[-1])
        right=np.clip(np.searchsorted(frame_times,targets),1,len(frame_times)-1)
        left=right-1
        w=(targets-frame_times[left])/(frame_times[right]-frame_times[left]+1e-10)
        v=cqt[:,left]*(1-w)[None,:]+cqt[:,right]*w[None,:]
        v=np.sqrt(np.maximum(v,0))
        v/=v.sum(axis=0,keepdims=True)+1e-8
        out[i]=v
    return centers,out

def rolling_inference(audio_path, top_roots=4):
    y=read_audio(audio_path)
    duration=len(y)/SR
    C,frame_times=extract_continuous_cqt72(y)
    centers,X=make_rolling_windows(C,frame_times,duration)
    p=predict_chord_posteriors(X,top_roots=top_roots)
    return {'centers':centers,'probabilities':p,'duration':duration,'step':STEP}

print('Continuous frontend ready. Call rolling_inference(path) when audio is available.')

## 6. Training the harmonic language model (real progressions only)

The acoustic CNN predicts chord identity from sound. The harmonic model predicts what **chord EVENT** is likely next, given preceding **distinct events** (not preceding 200 ms frames).

The language model was trained on the **70 RWC training songs only**, not on GuitarSet/IDMT/shared isolated chord samples. Identical adjacent segment labels are merged into an event when contiguous (≤0.15 s gap). Annotation gaps break phrases; no transition is counted across a gap. For a candidate next chord `c_j=(r_j,q_j,s_j)` and preceding event `c_{j-1}`:

- Outcome `o=((r_j-r_{j-1}) mod 12, q_j, s_j)` (transposition-invariant).
- First-order context `h1=(q_{j-1},s_{j-1})`.
- Second-order context `h2=((r_{j-1}-r_{j-2}) mod 12,q_{j-2},s_{j-2},q_{j-1},s_{j-1})`.

**Counts and interpolated smoothing**: global additive smoothing `α=0.05`; first-order context weight `n1/(n1+30)`; second-order context weight `n2/(n2+20)`. An unseen history backs off to first-order/global. These numbers were saved with the model and must be loaded, not guessed.

Observed training counts: **8,619** chord events, **8,096** transitions, **7,639** second-order triples; **349** second-order contexts. On validation next-event prediction, perplexity first-order **33.38**, second-order **30.37**. This measures harmonic prediction only, not full audio accuracy.

In [ ]:
# Rebuild the saved LM from RWC annotations when features are available.
# This cell defines the procedure; it DOES NOT overwrite the trusted saved model.
from collections import Counter, defaultdict

def labeled_phrases(npz_path, gap_tolerance=0.15):
    with np.load(npz_path,allow_pickle=False) as z:
        r=z['root'].astype(int); q=z['quality'].astype(int)
        s=z['seventh'].astype(int); starts=z['t_start'].astype(float); ends=z['t_end'].astype(float)
    phrases=[]; phrase=[]
    for i in np.argsort(starts):
        a,b=float(starts[i]),float(ends[i]); state=(int(r[i]),int(q[i]),int(s[i]))
        if b<=a: continue
        if phrase and a-phrase[-1]['end']>gap_tolerance:
            phrases.append(phrase); phrase=[]
        if phrase and phrase[-1]['chord']==state and abs(a-phrase[-1]['end'])<=gap_tolerance:
            phrase[-1]['end']=max(phrase[-1]['end'],b)
        else:
            phrase.append({'chord':state,'start':a,'end':b})
    if phrase: phrases.append(phrase)
    return phrases

def fit_language_counts(train_song_ids, feature_dir):
    c0=Counter(); c1=defaultdict(Counter); c2=defaultdict(Counter); durations=[]
    for song in train_song_ids:
        path=feature_dir/f'RWC_P{int(song):03d}.npz'
        for phrase in labeled_phrases(path):
            chords=[e['chord'] for e in phrase]
            durations += [e['end']-e['start'] for e in phrase]
            for j in range(1,len(chords)):
                older=chords[j-2] if j>=2 else None
                prev,nxt=chords[j-1],chords[j]
                outcome=((nxt[0]-prev[0])%12,nxt[1],nxt[2])
                c0[outcome]+=1
                c1[(prev[1],prev[2])][outcome]+=1
                if older is not None:
                    ctx=((prev[0]-older[0])%12,older[1],older[2],prev[1],prev[2])
                    c2[ctx][outcome]+=1
    return {'counts_global':c0,'counts_first_order':dict(c1),
            'counts_second_order':dict(c2),'durations_seconds':np.asarray(durations)}

if FEATURES_DIR.is_dir() and all((FEATURES_DIR/f'RWC_P{int(s):03d}.npz').exists() for s in manifest['splits']['train']):
    reconstructed=fit_language_counts(manifest['splits']['train'], FEATURES_DIR)
    for k in ('counts_global','counts_first_order','counts_second_order'):
        print(k, 'matches saved:', reconstructed[k] == lm[k])
    print('duration sample count:',len(reconstructed['durations_seconds']))
else:
    print('RWC training feature cache not present locally; saved LM is sufficient for inference.')

## 7. Harmonic transition probabilities and durations

The HSMM uses **duration distributions**, not frame-level self-transition probabilities. If event `j` lasts `d` observation steps, we score `P(D_j=d)` once for the entire segment. A new event has a different base chord; no self-transition between identical chord states is needed.

The baseline duration prior is a log-normal fitted to the training annotation durations. Training median **~1.658 s**, mean **~1.850 s**, 10th/90th percentiles **~0.796 s / ~3.121 s**. Current implementation constrains candidates to `0.4–6.0 s` with a special allowance for the opening segment.

The practical score is a **weighted decoding potential**, not a fully normalized generative HSMM probability:

\[
\operatorname{score}(\text{path})=\sum_{j}\left[\lambda_A\sum_{t\in\text{segment }j}\log(288 P_{CNN}(z_j\mid x_t))
+\lambda_D\log P(D_j=d_j) + \lambda_L\log P(z_j\mid z_{j-1},z_{j-2})\right].
\]

Historical prototype parameters: `STEP=0.2`, `λ_A=0.28`, `λ_L=0.35`, `λ_D=0.50`, duration 0.4–6.0s, beam 30, top-6 states per candidate segment. These were **not exhaustively tuned**. Overlapping CNN windows mean their evidence is correlated, motivating acoustic tempering.

In [ ]:
from functools import lru_cache
C0=lm['counts_global']; C1=lm['counts_first_order']; C2=lm['counts_second_order']
P0=lm['global_prior']; TAU1=float(lm['tau1']); TAU2=float(lm['tau2'])

def decode_state(s):
    s=int(s)
    return s//(Q*S), (s//S)%Q, s%S

def encode_state(r,q,s): return int(r)*(Q*S)+int(q)*S+int(s)

@lru_cache(maxsize=200000)
def transition_log_probability(older,previous,nxt,order=2):
    prev=decode_state(previous); candidate=decode_state(nxt)
    o=((candidate[0]-prev[0])%12,candidate[1],candidate[2])
    p=float(P0[o])
    c=C1.get((prev[1],prev[2]))
    if c:
        n=sum(c.values()); w=n/(n+TAU1)
        p=w*c[o]/n+(1-w)*p
    if order>=2 and older>=0:
        old=decode_state(older)
        ctx=((prev[0]-old[0])%12,old[1],old[2],prev[1],prev[2])
        c=C2.get(ctx)
        if c:
            n=sum(c.values());w=n/(n+TAU2)
            p=w*c[o]/n+(1-w)*p
    return float(np.log(max(p,1e-12)))

D=np.asarray(lm['durations_seconds'],dtype=float)
D=D[np.isfinite(D)&(D>0)]
DUR_SHAPE,_,DUR_SCALE=lognorm.fit(D,floc=0)
def duration_log_probability(n_steps,step=STEP):
    duration=n_steps*step
    lower=max(0,duration-step/2); upper=duration+step/2
    prob=lognorm.cdf(upper,DUR_SHAPE,scale=DUR_SCALE)-lognorm.cdf(lower,DUR_SHAPE,scale=DUR_SCALE)
    return float(np.log(max(prob,1e-12)))

print('Example log-transition:',transition_log_probability(-1,encode_state(0,0,0),encode_state(7,0,1),1))
print('Duration prior median:',np.median(D))

## 8. Approximate second-order HSMM via segmental Viterbi

At each candidate **segment end time** `t`, consider segment durations `d` and start `t-d`. The segment acoustic score is computed efficiently with prefix sums over all 288 log-posteriors. For each segment, only the top `K=6` acoustically supported chord candidates are expanded. Dynamic-programming hypotheses are keyed by the most recent one or two **chord events**. At each observation endpoint retain at most `BEAM=30` best hypotheses. The final path is reconstructed by backtracking stored pointers.

Trade-offs:

- **Pros:** jointly predicts boundary times, chord labels and durations, while using two-event context.
- **Cons:** beam and top-K pruning can exclude the right path; no global optimality guarantee. The duration prior discourages very short repeated segments. Boundary positions are quantized to 0.2 s.
- **Known edge cases:** truncated duration `6 s` can split genuinely sustained longer chords. `N` regions and inversions are not represented. Avoid treating `P(CNN|audio)` as calibrated generative emissions without calibration.

Codex should preserve this decoder as a **baseline**, then add unit tests, batch/profiling and controlled optimizations before changing it.

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class DecoderConfig:
    step:float=0.2
    acoustic_weight:float=0.28
    lm_weight:float=0.35
    duration_weight:float=0.5
    min_duration:float=0.4
    max_duration:float=6.0
    beam_size:int=30
    segment_top_k:int=6

CFG=DecoderConfig()

def segmental_viterbi(probabilities,order=2,config=CFG):
    assert order in (1,2)
    probs=np.asarray(probabilities,dtype=float)
    n=len(probs)
    assert probs.ndim==2 and probs.shape[1]==N_STATES
    if n==0:return []
    logp=np.log(np.maximum(probs,1e-8))+np.log(N_STATES)
    logp*=config.acoustic_weight
    prefix=np.vstack([np.zeros((1,N_STATES)),np.cumsum(logp,axis=0)])
    min_steps=max(1,round(config.min_duration/config.step))
    max_steps=max(min_steps,round(config.max_duration/config.step))
    dp=[{} for _ in range(n+1)]
    dp[0][()]=(0.0,None,None,None)
    for end in range(1,n+1):
        possibilities={}
        lengths=list(range(min_steps,min(max_steps,end)+1))
        if end<=max_steps and end not in lengths:lengths.append(end)
        for d in lengths:
            start=end-d
            if start<0 or not dp[start]:continue
            segment=prefix[end]-prefix[start]
            candidates=np.argsort(segment)[-config.segment_top_k:][::-1]
            dur=config.duration_weight*duration_log_probability(d,config.step)
            for oldkey,(oldscore,*_) in dp[start].items():
                previous=oldkey[-1] if oldkey else None
                older=oldkey[-2] if len(oldkey)>=2 else -1
                for candidate in candidates:
                    candidate=int(candidate)
                    if previous is not None and candidate==previous:continue
                    score=oldscore+float(segment[candidate])+dur
                    if previous is not None:
                        score+=config.lm_weight*transition_log_probability(
                            older,previous,candidate,order)
                    key=(candidate,) if order==1 else ((previous,candidate) if previous is not None else (candidate,))
                    old=possibilities.get(key)
                    if old is None or score>old[0]:
                        possibilities[key]=(score,start,oldkey,candidate)
        if possibilities:
            dp[end]=dict(sorted(possibilities.items(),key=lambda kv:kv[1][0],reverse=True)[:config.beam_size])
    if not dp[n]:raise RuntimeError('No complete decoding path; check durations/candidates')
    key=max(dp[n],key=lambda k:dp[n][k][0])
    events=[]; end=n
    while end>0:
        score,start,previous_key,state=dp[end][key]
        events.append((start,end,int(state)))
        end,key=start,previous_key
    events.reverse()
    assert events[0][0]==0 and events[-1][1]==n
    return events

def expand_events(events,n):
    out=np.full(n,-1,dtype=int)
    for a,b,s in events:out[a:b]=s
    return out

ROOT_NAMES=['C','Db','D','Eb','E','F','Gb','G','Ab','A','Bb','B']
def chord_name(state):
    root,q,s=decode_state(state)
    qual={'maj':'','min':'m','dim':'dim','aug':'aug','sus2':'sus2','sus4':'sus4'}[QUALITY[q]]
    seventh={'none':'','b7':'7','maj7':'maj7','dim7':'dim7'}[SEVENTH[s]]
    return ROOT_NAMES[root]+qual+seventh

def events_dataframe(events,config=CFG):
    return pd.DataFrame([
        {'start_s':a*config.step,'end_s':b*config.step,
         'duration_s':(b-a)*config.step,'state':s,'chord':chord_name(s)}
        for a,b,s in events
    ])
print('HSMM ready. Note: decoder is approximate beam-pruned segmental Viterbi.')

## 9. Transcribe an audio file

Run when the repository has a legal local audio file. To avoid repeatedly recomputing audio features, cache `centers`, `probabilities`, `duration`, `step`, plus a **pipeline version and audio checksum**. Old Colab caches were keyed mostly by numeric song ID and `step` only, making ID collisions and stale preprocessing a risk.

The RWC audio used earlier was accessed through a remote ZIP; a local CLI should instead take an explicit `--audio /path/to/song.mp3`, never rely on `RWC_PNNN` naming for arbitrary songs.

In [ ]:
# Change to a real audio path, or leave None to skip.
AUDIO_PATH = None  # e.g. Path('data/audio/Hey Jude.mp3')

if AUDIO_PATH is not None:
    AUDIO_PATH=Path(AUDIO_PATH)
    features=rolling_inference(AUDIO_PATH,top_roots=4)
    p=features['probabilities']
    print('Windows:',len(p),'duration:',features['duration'])
    for order in (1,2):
        t0=time.time()
        events=segmental_viterbi(p,order=order)
        df=events_dataframe(events)
        df['end_s']=np.minimum(df['end_s'],features['duration'])
        outfile=AUDIO_PATH.with_name(AUDIO_PATH.stem+f'_hsmm_order{order}.csv')
        df.to_csv(outfile,index=False)
        print(f'Order {order}: {len(events)} events, {time.time()-t0:.1f}s → {outfile}')
        display(df.head(12))
else:
    print('Set AUDIO_PATH to run full-song transcription.')

## 10. Validation and evaluation — don't confuse three different tests

**A. Segment-aligned CNN checkpoint evaluation:** known reference chord segments are supplied to the CNN. Historical validation RQS **70.96%** (`~70.91%` with a truncated joint-probability evaluator). This is *not* deployment.

**B. Oracle-event decoding experiment:** true changes define event boundaries; model predicts only chord identities. CNN **70.86%**, best first-order **72.54%**, best second-order **72.27%** on annotated segments. This diagnostic leaks *boundary locations* by design; do not present as end-to-end accuracy.

**C. Full continuous-audio evaluation:** no annotated boundaries are passed to decoder. Correctly evaluates the system we intend to ship. On 15 RWC validation songs:

| Model | Root | Root+quality | Full RQS | Boundary F1 (±0.5s) |
|---|---:|---:|---:|---:|
| CNN per 200 ms | 77.92% | 70.33% | 58.67% | 0.557 |
| 1st-order HSMM | 78.55% | 73.72% | 64.87% | 0.852 |
| **2nd-order HSMM** | **78.60%** | **74.10%** | **65.45%** | **0.852** |

Boundary F1 measured matched changes within ±0.5 s, with reference gaps excluded; second-order precision **0.825**, recall **0.880**, matched-boundary MAE **~131 ms**. Song-bootstrap second-order RQS improvement over CNN **+6.78 pp** (95% interval **+4.56 to +8.91 pp**). The 0.58 pp second-order vs first-order difference is *not* established as significant.

**Separate root/quality/seventh errors**. Root+quality may be correct while seventh is wrong. Metrics on the 200 ms grid approximate time weighting. For Beatles Isophonics comparisons, map labels to Harte convention and use `mir_eval.chord` metrics; check MP3-to-annotation alignment before comparing with literature.

In [ ]:
# Quantitative scorer for continuous RWC validation audio (not applicable to unlabeled MP3s).
def rwc_reference_at_grid(npz_path, centers):
    with np.load(npz_path,allow_pickle=False) as z:
        starts=z['t_start'].astype(float); ends=z['t_end'].astype(float)
        states=z['root'].astype(int)*Q*S+z['quality'].astype(int)*S+z['seventh'].astype(int)
    order=np.argsort(starts); starts=starts[order]; ends=ends[order]; states=states[order]
    ix=np.searchsorted(starts,centers,side='right')-1
    si=np.clip(ix,0,len(starts)-1)
    valid=(ix>=0)&(centers>=starts[si])&(centers<ends[si])
    truth=np.full(len(centers),-1,dtype=int);truth[valid]=states[si[valid]]
    return truth,valid

def rqs_metrics(pred,truth,valid):
    p=np.asarray(pred)[valid];t=np.asarray(truth)[valid]
    if not len(p):return {'root':np.nan,'root_quality':np.nan,'rqs':np.nan}
    root=p//(Q*S)==t//(Q*S)
    quality=(p//S)%Q==(t//S)%Q
    return {'root':float(root.mean()),'root_quality':float((root&quality).mean()),
            'rqs':float((p==t).mean())}

# Example (requires an existing rolling cache):
# z=np.load('rolling_predictions/RWC_P080_rolling.npz')
# truth,mask=rwc_reference_at_grid(FEATURES_DIR/'RWC_P080.npz',z['centers'])
# cnn=z['probabilities'].argmax(axis=1)
# hsmm=expand_events(segmental_viterbi(z['probabilities'],order=2),len(cnn))
# print(rqs_metrics(cnn,truth,mask), rqs_metrics(hsmm,truth,mask))
print('Continuous RWC evaluation helper ready.')

## 11. Visualize and listen: error analysis

Different errors require different fixes. If the correct chord is low-ranked by CNN, work on audio preprocessing, posterior calibration and training data. If CNN ranks it well but HSMM overrides it, investigate transition/duration weights and candidate pruning. If identity is correct but timestamps are shifted, inspect the CQT window size, beat-aware prior and 200 ms quantization. Listen while comparing reference vs CNN vs first/second HSMM when annotations exist.

Do **not** claim external-song accuracy based on subjective listening alone. The Beatles tracks used in the prototype were *Hey Jude*, *Something*, *The Long and Winding Road*, and a separate Bill Withers recording *Just the Two of Us*. Their predictions are interesting demonstrations but require matched external labels for quantitative scores.

In [ ]:
import matplotlib.pyplot as plt

def plot_transcription_rows(predictions, start=0, length=20, step=STEP):
    # predictions maps readable row name to 1-D integer state array.
    fig,ax=plt.subplots(figsize=(15, max(2.6,0.85*len(predictions)+1)))
    end=start+length
    for row,(title,states) in enumerate(predictions.items()):
        states=np.asarray(states)
        change=np.r_[0,np.flatnonzero(states[1:]!=states[:-1])+1,len(states)]
        for a,b in zip(change[:-1],change[1:]):
            lo=max(a*step,start);hi=min(b*step,end)
            if hi<=lo:continue
            state=int(states[a]);color=plt.cm.tab20(state%20) if state>=0 else '#eeeeee'
            ax.broken_barh([(lo,hi-lo)],(row,0.70),facecolors=color,edgecolors='white')
            if hi-lo>0.7 and state>=0:
                ax.text((lo+hi)/2,row+0.35,chord_name(state),ha='center',va='center',fontsize=8)
    ax.set_xlim(start,end);ax.set_ylim(-.1,len(predictions)+.1)
    ax.invert_yaxis();ax.set_yticks(np.arange(len(predictions))+.35,list(predictions))
    ax.set_xlabel('Song time (seconds)');ax.grid(axis='x',alpha=.2)
    plt.show()

# Use with e.g. {'CNN':cnn, 'Second-order':hsmm}. This is a lightweight
# notebook diagnostic; the continuous JavaScript playback viewer lives in the app/UI layer.

## 12. Engineering acceptance tests for Codex

The notebook's purpose is to enable clean repo migration. **Codex should not silently change model behavior.** Suggested required tests:

1. Checkpoint loads **strictly** on CPU and GPU (where available), 58 saved state entries, no missing keys.
2. Known `72×32` feature sample → reproducible 12-root, 6-quality, 4-seventh logits and 288-state posterior; posterior rows sum to 1.
3. Root-relative rotation has correct orientation for all 12 roots, preserves shape and norms, and rotates identically in training/inference.
4. Preprocessing regression: compare rolling CQT72 tensors between the saved feature format and the local implementation, not just final chord accuracy.
5. Compare exact rolling posterior matrix for one saved `RWC_PNNN` fixture within numeric tolerance (CPU/GPU differences documented).
6. Verify reconstructed harmonic counts against saved counters; 70-song training split only, gap rule and merging consistent.
7. First- and second-order transition distributions are valid (sum over 288 outcomes to ~1 for each history), including unseen-history backoff.
8. Duration probabilities finite and sane; decoder produces **full contiguous coverage** of the observation grid; adjacent decoded chord events have different base chord IDs.
9. Decoder compares to saved validation fixtures; reproduce 65.45% RQS within known tolerance. **Do not run the untouched test set until implementation and hyperparameters are frozen.**
10. No hard-coded numeric song IDs for user input; audio hash, model version and frontend hash govern cache keys.
11. Add export formats `.csv` and `.json`, plus a CLI entrypoint `python -m chordrec.cli --audio ...`.
12. Report root, root+quality, RQS and ±0.5 s boundary F1 with duration weighting / gap policies documented.

In [ ]:
# Simple deterministic HSMM smoke test using a synthetic chord sequence.
# This checks coverage, not model accuracy.
rng=np.random.default_rng(42)
test=np.full((40,N_STATES),1e-5,dtype=np.float64)
a=encode_state(0,0,0);b=encode_state(7,0,1)
test[:20,a]=.92; test[20:,b]=.92
test/=test.sum(axis=1,keepdims=True)
for order in (1,2):
    events=segmental_viterbi(test,order=order)
    assert events[0][0]==0 and events[-1][1]==40
    assert all(events[i][1]==events[i+1][0] for i in range(len(events)-1))
    assert all(events[i][2]!=events[i+1][2] for i in range(len(events)-1))
    print('order',order, 'events:',[(x,y,chord_name(z)) for x,y,z in events])

## 13. Future improvements — after reproducibility

**Recommended order:** (1) verify the frontend against the saved training features, (2) inspect posterior calibration / test all-root vs top-root enumeration and segment candidate pruning, (3) tune decoder weights on validation only and examine failure songs, (4) consider tempo/beat-aware chord duration models and local key awareness, (5) add well-licensed real progression data to the **harmonic** model only if a controlled evaluation justifies it, (6) expand chord vocabulary and train extension recognition with sufficient positive samples.

Avoid adding more model classes prematurely: the present system already shows clear improvements over CNN-only decoding. Report any new gain with paired song-level metrics and an untouched test benchmark at the end.

### Specific action requested of Codex

> Keep the implementation in clear Python modules. Use the **existing checkpoints** and **exact vocabulary/order**. The local audio-to-CSV/JSON inference CLI, cache, CNN training script, harmonic transition model, and synchronized chord/audio viewer are part of this repository. Never train the harmonic transition model on isolated-chord datasets. Preserve the distinction between training, validation and test songs.

**This notebook does not include pretrained weight bytes, copyrighted music recordings, or the original CNN training experiment's complete configuration.** Those must be supplied separately to the local repository.